# Retrain "marvin" v2: fewer false triggers (hard negatives + the ESP32's own recordings)

The first model (`marvin.tflite`) detects "Marvin" well, but on the ESP32 it also fires on **sound-alike words**:
"Martin", "marble", "Marvel", "carving", "Mark", "Markovnikov". On the device those reach the same score as a real
"Marvin", so no threshold can fix it: the model has to learn the difference. The first run was trained with
**no** sound-alike negatives (`CONFUSABLE_PHRASES = []`).

### What is different from the first run
| | first run | this run (v2) |
|---|---|---|
| Sound-alike words ("martin", "marble", ...) | none | 16 words x 400 synthetic voices, in their **own** training set with their own weight (Step 10) |
| Real false triggers recorded by the ESP32 | none | the device's recordings that Whisper labelled as NOT "Marvin", each used 10x |
| Real "Marvin" recorded by the ESP32 | none | used as an honest **test** of the new model (never trained on) |
| Test on the device's own recordings | none | **Step 12b**: old vs new model, with exactly the firmware's pipeline |
| Output folder | `microwakeword/marvin` | `microwakeword/marvin_v2` (a new folder: reusing the old one would *continue* the old model) |

Everything else (datasets, augmentation, model size, training recipe, export format) is the same, so the result
drops straight into the firmware.

### Before you start
1. On your PC, open the folder `marvin_retrain_v2` (next to this notebook). It has the files Step 2b asks for:
   `device_positives.zip`, `device_hard_negatives.zip` and `old_marvin.tflite` (the current model, for comparison).
2. In Colab: `Runtime → Change runtime type → T4 GPU`.
3. **Run the cells from top to bottom**, one at a time. Step 1 restarts the runtime (expected); then continue with Step 2.
4. Budget **2.5–4 hours** (downloads ~20 min, sound-alike voices ~15 min, features ~30 min, training 15,000 steps ~2–3 h).
   Keep the browser tab open. If Colab disconnects, see the note in Step 11: it resumes from the last checkpoint.
5. At the end, `marvin_export.zip` downloads. Put its `marvin.tflite` + `marvin.json` into `kws_s3/model/` and rebuild.

> Data licences: Speech Commands is CC-BY-4.0. The negative feature sets and some noise sets are **non-commercial** (CC-BY-NC).

## Step 1: Install everything (≈3–5 min)
The runtime **restarts automatically** at the end ("Your session crashed" is expected). After that, continue with Step 2 and **don't** re-run Step 1.

In [ ]:
import os, sys
print("Python", sys.version.split()[0])
if sys.version_info[:2] > (3, 12):
    print("⚠️  This notebook was tested on Python 3.12. If an install below fails, pick an older runtime:\n"
          "   Runtime → Change runtime type → Runtime version → 2026.07 (Python 3.12)")
%cd /content

# microWakeWord (the training framework) and Piper sample generator (synthetic voices), pinned to tested versions
!test -d microWakeWord || git clone -q https://github.com/kahrendt/microWakeWord.git
!cd microWakeWord && git checkout -q 4665173cd35f1cff9a61e06fc427f124766c488e
!test -d piper-sample-generator || git clone -q https://github.com/rhasspy/piper-sample-generator.git
!cd piper-sample-generator && git checkout -q 2971426a55072f7d22fec416ca7800df8bd23207

# audio-metadata fork: unpins a package that breaks Jupyter
!pip install -q 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
# datasets<4: newer versions need torchcodec to decode audio. audiomentations>=0.40 provides AddColorNoise.
!pip install -q -e ./microWakeWord "datasets==3.6.0" "audiomentations>=0.40" librosa soundfile
# --no-deps: Piper's generator pins an old audiomentations that breaks microWakeWord, so install it without deps
!pip install -q --no-deps -e ./piper-sample-generator
!pip install -q "piper-tts==1.3.0" webrtcvad-wheels
!test -f piper-sample-generator/models/en_US-libritts_r-medium.pt || wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt

print("\n✅ Install finished. Restarting the runtime now (this is expected); then run Step 2.")
os.kill(os.getpid(), 9)

## Step 2: Settings (already filled in for this run)
Nothing needs changing. The knobs, if Step 12b shows a problem:
* **Still fires on sound-alikes** → raise `HARD_NEG_WEIGHT` (e.g. 5) or `CONFUSABLE_SAMPLES` (e.g. 800) and retrain.
* **Misses real "Marvin" more than before** → lower `HARD_NEG_WEIGHT` (e.g. 2), or remove the closest phrases ("garvin", "harvin").
* Use a **new** `RUN_NAME` for every retraining, otherwise Step 11 continues the previous run.

In [ ]:
MODE = "speech_commands"        # "speech_commands" = practise with a word from the dataset
                                # "custom"          = your own keyword (e.g. the SIH keyword)
SC_WORD = "marvin"              # used when MODE == "speech_commands"
CUSTOM_WAKE_WORD = "hey vyom"   # used when MODE == "custom"; spell it how it SOUNDS
RUN_NAME = "marvin_v2"          # output folder in Drive. Training RESUMES from checkpoints in this folder,
                                # so use a new name for every new model (never the old model's folder)

TTS_SAMPLES = 2000              # synthetic Piper voices of the keyword
CONFUSABLE_PHRASES = [          # sound-alikes that must NOT trigger. The first eight fooled the ESP32 on 27 Sep.
    "martin", "marvel", "marble", "carving", "mark", "markovnikov", "pardon", "can you",
    "margin", "marlin", "martian", "marvelous", "garvin", "harvin", "starving", "carbon",
]
CONFUSABLE_SAMPLES = 400        # Piper voices per phrase (16 x 400 = 6,400 clips, about 10-20 min on a T4)
HARD_NEG_WEIGHT = 3.0           # how often training batches draw from the sound-alikes (Step 10). Higher = fewer
                                # false triggers on similar words, but check the misses in Step 12b
MAX_SC_NEGATIVES = 20000        # other Speech Commands words used as "not the keyword"
TRAINING_STEPS = 15000          # the first run used 10000 (about 1-2 h); more data needs a few more steps
SAVE_TO_DRIVE = True            # keep the trained model in Google Drive (survives disconnects)
DOWNLOAD_EXTRA_NOISE = True     # music (FMA) + everyday sounds (AudioSet) for augmentation

MY_RECORDINGS_ZIP = ""          # set by Step 2b: device_positives.zip (real "Marvin" through the ESP32's microphone)
MY_NEGATIVES_ZIP = ""           # set by Step 2b: device_hard_negatives.zip (what fooled the ESP32)

# ---------------- nothing to edit below this line ----------------
import re
SC_WORDS = ["backward","bed","bird","cat","dog","down","eight","five","follow","forward","four","go","happy",
            "house","learn","left","marvin","nine","no","off","on","one","right","seven","sheila","six","stop",
            "three","tree","two","up","visual","wow","yes","zero"]
assert MODE in ("speech_commands", "custom"), "MODE must be 'speech_commands' or 'custom'"
if MODE == "speech_commands":
    assert SC_WORD in SC_WORDS, f"SC_WORD must be one of: {', '.join(SC_WORDS)}"
WAKE_WORD = SC_WORD if MODE == "speech_commands" else CUSTOM_WAKE_WORD.strip()
MODEL_NAME = re.sub(r"[^a-z0-9]+", "_", WAKE_WORD.lower()).strip("_")
print(f"Wake word: '{WAKE_WORD}'   model name: {MODEL_NAME}   run: {RUN_NAME}   mode: {MODE}")

## Step 2b: Upload the ESP32's recordings
Click **Choose Files** and select, from the `marvin_retrain_v2` folder on your PC, all three files:
`device_positives.zip`, `device_hard_negatives.zip`, `old_marvin.tflite`.
They are short clips cut from what the ESP32 streamed to the cloud server, labelled by Whisper
("Marvin" or not). If you re-run this cell later it reuses the files already uploaded.

In [ ]:
import os
DEVICE_DIR = "/content/device_data"
os.makedirs(DEVICE_DIR, exist_ok=True)
TARGETS = {"positives": "device_positives.zip", "negatives": "device_hard_negatives.zip", ".tflite": f"old_{MODEL_NAME}.tflite"}
have = lambda name: os.path.exists(f"{DEVICE_DIR}/{name}")

if not (have("device_positives.zip") and have("device_hard_negatives.zip")):
    from google.colab import files
    for name, data in files.upload().items():
        for key, target in TARGETS.items():
            if key in name:
                open(f"{DEVICE_DIR}/{target}", "wb").write(data)
        if os.path.exists(name):
            os.remove(name)   # upload() also leaves a copy in the current folder

if have("device_positives.zip"):
    MY_RECORDINGS_ZIP = f"{DEVICE_DIR}/device_positives.zip"
if have("device_hard_negatives.zip"):
    MY_NEGATIVES_ZIP = f"{DEVICE_DIR}/device_hard_negatives.zip"
print("real 'Marvin' from the ESP32    :", MY_RECORDINGS_ZIP or "NOT uploaded (Step 12b will have less to test)")
print("ESP32 false triggers            :", MY_NEGATIVES_ZIP or "NOT uploaded (only synthetic sound-alikes will be used)")
print("old model for the comparison    :", f"{DEVICE_DIR}/old_{MODEL_NAME}.tflite" if have(f"old_{MODEL_NAME}.tflite") else "not uploaded (looked for in Drive in Step 3)")

## Step 3: Folders (and Google Drive)
Big temporary files go to the fast local disk `/content/work`. The trained model and the exports go to Drive (if enabled).

In [ ]:
import os, sys, glob, random, shutil, subprocess, json
WORK = "/content/work"
SC_DIR = "/content/speech_commands"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/microwakeword"
else:
    DRIVE_ROOT = "/content/output"
OUT = f"{DRIVE_ROOT}/{RUN_NAME}"
TRAIN_DIR = f"{OUT}/trained_model"
POS_DIR   = f"{WORK}/clips/positives"        # positives/{training,validation,testing}/*.wav
NEG_DIR   = f"{WORK}/clips/negatives"        # other Speech Commands words
HARD_DIR  = f"{WORK}/clips/hard_negatives"   # sound-alikes + the ESP32's real false triggers
FEAT_POS  = f"{WORK}/features/{RUN_NAME}/wakeword"
FEAT_NEG  = f"{WORK}/features/{RUN_NAME}/extra_negatives"
FEAT_HARD = f"{WORK}/features/{RUN_NAME}/hard_negatives"
HF_NEG    = f"{WORK}/negative_datasets"
RIR_DIR   = f"{WORK}/mit_rirs"
for d in (WORK, OUT):
    os.makedirs(d, exist_ok=True)

# the model to beat in Step 12b: uploaded in Step 2b, else the first run's export in Drive
OLD_MODEL = next((p for p in (f"{DEVICE_DIR}/old_{MODEL_NAME}.tflite", f"{DRIVE_ROOT}/{MODEL_NAME}/export/{MODEL_NAME}.tflite")
                  if os.path.exists(p)), None)
print("Work dir:", WORK, "\nOutput dir:", OUT, "\nOld model for Step 12b:", OLD_MODEL or "none found")
if os.path.isdir(TRAIN_DIR) and os.listdir(TRAIN_DIR):
    print(f"Note: {TRAIN_DIR} already has checkpoints, so Step 11 will CONTINUE that run "
          "(fine after a disconnect; for a brand-new model change RUN_NAME in Step 2).")

## Step 4: Get the Speech Commands dataset (≈2.3 GB, 2–4 min)
Downloading it straight into Colab is much faster than uploading your own copy.

In [ ]:
SC_URL = "http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz"
if not os.path.isdir(f"{SC_DIR}/marvin"):
    os.makedirs(SC_DIR, exist_ok=True)
    !wget -q --show-progress -O /content/sc.tgz {SC_URL}
    !tar -xzf /content/sc.tgz -C {SC_DIR} && rm /content/sc.tgz

counts = {w: len(glob.glob(f"{SC_DIR}/{w}/*.wav")) for w in SC_WORDS}
print(f"{sum(counts.values()):,} clips of 35 words (1 s, 16 kHz). Per word:")
print(", ".join(f"{w} {n}" for w, n in counts.items()))
print("Background noise files:", sorted(os.path.basename(f) for f in glob.glob(f"{SC_DIR}/_background_noise_/*.wav")))

## Step 5: Build the positive clips (the wake word)
Sources:
1. **Real voices:** the Speech Commands recordings of `SC_WORD` (practice mode) and/or **your own recordings** (`MY_RECORDINGS_ZIP`). The dataset's official speaker split is kept, so validation and test use **speakers the model never trained on**.
2. **Synthetic voices:** Piper TTS says the keyword with hundreds of different speakers, speeds and styles.

Every clip is converted to 16 kHz mono and the silence at the start and end is trimmed, so the word ends close to the end of the clip. That matters because the model learns to fire right after the word ends.

In [ ]:
import numpy as np, librosa, soundfile as sf

def save_trimmed(src, dst, top_db=30):
    """Load any audio file -> 16 kHz mono, trim leading/trailing silence, save as 16-bit wav. Returns False if unusable."""
    try:
        y, _ = librosa.load(src, sr=16000, mono=True)
    except Exception as e:
        print("  skip (cannot read):", src, e); return False
    if y.size == 0 or np.abs(y).max() < 1e-4:
        return False
    y, _ = librosa.effects.trim(y, top_db=top_db)
    if len(y) < 0.2 * 16000:          # shorter than 0.2 s after trimming -> probably no speech
        return False
    y = y / max(1e-6, np.abs(y).max()) * 0.9
    sf.write(dst, y, 16000, subtype="PCM_16")
    return True

def sc_split(word):
    """Speech Commands' official speaker-independent split."""
    val = set(open(f"{SC_DIR}/validation_list.txt").read().split())
    test = set(open(f"{SC_DIR}/testing_list.txt").read().split())
    out = {"training": [], "validation": [], "testing": []}
    for f in sorted(glob.glob(f"{SC_DIR}/{word}/*.wav")):
        rel = f"{word}/{os.path.basename(f)}"
        out["validation" if rel in val else "testing" if rel in test else "training"].append(f)
    return out

def split_list(files, fractions=(0.9, 0.05, 0.05), seed=0):
    files = sorted(files); random.Random(seed).shuffle(files)
    a = int(len(files) * fractions[0]); b = a + int(len(files) * fractions[1])
    return {"training": files[:a], "validation": files[a:b], "testing": files[b:]}

def add_clips(split_dict, dst_root, prefix):
    n = 0
    for split, files in split_dict.items():
        os.makedirs(f"{dst_root}/{split}", exist_ok=True)
        for i, f in enumerate(files):
            n += save_trimmed(f, f"{dst_root}/{split}/{prefix}_{i:05d}.wav")
    return n

shutil.rmtree(POS_DIR, ignore_errors=True)
for s in ("training", "validation", "testing"):
    os.makedirs(f"{POS_DIR}/{s}", exist_ok=True)

# 5a. real recordings from Speech Commands (practice mode)
if MODE == "speech_commands":
    n = add_clips(sc_split(SC_WORD), POS_DIR, f"sc_{SC_WORD}")
    print(f"Speech Commands '{SC_WORD}': {n} real clips")

# 5b. your own recordings (optional, any mode). Few files -> used for validation/testing only (honest scores).
if MY_RECORDINGS_ZIP:
    rec_dir = f"{WORK}/my_recordings"
    shutil.rmtree(rec_dir, ignore_errors=True); os.makedirs(rec_dir)
    subprocess.run(["unzip", "-q", "-o", MY_RECORDINGS_ZIP, "-d", rec_dir], check=True)
    recs = [f for f in glob.glob(f"{rec_dir}/**/*", recursive=True)
            if f.lower().endswith((".wav", ".mp3", ".m4a", ".ogg", ".flac", ".aac", ".opus", ".webm"))]
    fr = (0.0, 0.5, 0.5) if len(recs) < 100 else (0.6, 0.2, 0.2)
    n = add_clips(split_list(recs, fr), POS_DIR, "myrec")
    print(f"Your recordings: {n} clips (split {fr})")

In [ ]:
# 5c. synthetic voices with Piper (≈1-3 min per 1000 samples on a T4 GPU)
PSG = "/content/piper-sample-generator"
PIPER_MODEL = f"{PSG}/models/en_US-libritts_r-medium.pt"

def tts(text, n, out_dir, batch=100):
    """Generate n wav files of `text` spoken by random Piper voices (skips if they already exist)."""
    os.makedirs(out_dir, exist_ok=True)
    if len(glob.glob(f"{out_dir}/*.wav")) >= n:
        print(f"  '{text}': {n} samples already there"); return
    cmd = [sys.executable, "-m", "piper_sample_generator", text, "--model", PIPER_MODEL,
           "--max-samples", str(n), "--batch-size", str(min(batch, n)), "--max-speakers", "800",
           "--length-scales", "0.75", "0.9", "1.0", "1.1", "1.25", "1.4",
           "--output-dir", out_dir]
    subprocess.run(cmd, cwd=PSG, check=True)
    print(f"  '{text}': generated {len(glob.glob(f'{out_dir}/*.wav'))} samples")

tts_dir = f"{WORK}/tts/{MODEL_NAME}"
tts(WAKE_WORD, TTS_SAMPLES, tts_dir)
n = add_clips(split_list(glob.glob(f"{tts_dir}/*.wav")), POS_DIR, "tts")
print(f"TTS positives added: {n}")

for s in ("training", "validation", "testing"):
    print(f"positives/{s}: {len(glob.glob(f'{POS_DIR}/{s}/*.wav'))} clips")
assert len(glob.glob(f"{POS_DIR}/training/*.wav")) > 0, "No training positives. Check Step 5 settings."

In [ ]:
# 5d. LISTEN to a few positives. If the TTS sounds wrong, change the spelling in Step 2 and re-run Step 5.
from IPython.display import Audio, display
for f in random.sample(glob.glob(f"{POS_DIR}/training/*.wav"), 3):
    print(os.path.basename(f)); display(Audio(f))

## Step 5e: Negative clips ("NOT the wake word")
* **Other Speech Commands words:** real people saying real words that must be ignored (as in the first run).
* **Hard negatives, in their own set** (new): the sound-alike phrases from Step 2 (Piper voices) and the ESP32's real
  false triggers. A separate set gets its own sampling weight in Step 10; mixed into the 20,000 clips above they
  would be only ~2% of what the model trains on, which is why the first model never learned the difference.
* The ESP32 recordings are split **by recording** (60/20/20), so Step 12b can test on false triggers the model never saw.

In [ ]:
# 5e-1. other Speech Commands words (same as the first run)
shutil.rmtree(NEG_DIR, ignore_errors=True)
neg_words = [w for w in SC_WORDS if not (MODE == "speech_commands" and w == SC_WORD)]
neg = {"training": [], "validation": [], "testing": []}
for w in neg_words:
    for s, files in sc_split(w).items():
        neg[s] += files
caps = {"training": MAX_SC_NEGATIVES, "validation": MAX_SC_NEGATIVES // 10, "testing": MAX_SC_NEGATIVES // 10}
for s in neg:
    random.Random(1).shuffle(neg[s]); neg[s] = neg[s][:caps[s]]
    os.makedirs(f"{NEG_DIR}/{s}", exist_ok=True)
    for i, f in enumerate(neg[s]):          # symlinks: fast, no copy needed (these are 16 kHz already)
        os.symlink(f, f"{NEG_DIR}/{s}/sc_{i:06d}.wav")
    print(f"negatives/{s}: {len(neg[s])} Speech Commands clips")

# 5e-2. sound-alike phrases (Piper voices), into the separate hard-negative set
shutil.rmtree(HARD_DIR, ignore_errors=True)
for s in ("training", "validation", "testing"):
    os.makedirs(f"{HARD_DIR}/{s}", exist_ok=True)
for k, phrase in enumerate(CONFUSABLE_PHRASES):
    d = f"{WORK}/tts/confusable_" + re.sub(r"[^a-z0-9]+", "_", phrase.lower()).strip("_")
    tts(phrase, CONFUSABLE_SAMPLES, d)
    add_clips(split_list(glob.glob(f"{d}/*.wav"), (0.8, 0.1, 0.1)), HARD_DIR, f"conf{k:02d}")

# 5e-3. the ESP32's real false triggers: few, so each is used 10x (every copy gets a different augmentation in
# Step 8). Split by recording, the same way for every copy, so held-out recordings stay unseen.
DEVICE_NEG_SPLIT = {"training": [], "validation": [], "testing": []}
if MY_NEGATIVES_ZIP:
    d = f"{WORK}/device_negatives"
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
    subprocess.run(["unzip", "-q", "-o", MY_NEGATIVES_ZIP, "-d", d], check=True)
    DEVICE_NEG_SPLIT = split_list(glob.glob(f"{d}/**/*.wav", recursive=True), (0.6, 0.2, 0.2))
    for r in range(10):
        add_clips(DEVICE_NEG_SPLIT, HARD_DIR, f"dev{r}")
    print("ESP32 false triggers (recordings):", {s: len(f) for s, f in DEVICE_NEG_SPLIT.items()}, "each used 10x")
print("hard negatives:", {s: len(os.listdir(f"{HARD_DIR}/{s}")) for s in ("training", "validation", "testing")})

## Step 6: Download room echoes and background noise for augmentation (≈5–15 min)
* **MIT room impulse responses:** makes clean speech sound like it was recorded in a real room.
* **Background noise:** the Speech Commands noise files, plus (optional) music from FMA and everyday sounds from AudioSet.

In [ ]:
import datasets, scipy.io.wavfile
from pathlib import Path
from tqdm.auto import tqdm

if not os.path.isdir(RIR_DIR) or not os.listdir(RIR_DIR):
    os.makedirs(RIR_DIR, exist_ok=True)
    rir = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    for row in tqdm(rir, desc="MIT RIRs"):
        name = row["audio"]["path"].split("/")[-1]
        scipy.io.wavfile.write(f"{RIR_DIR}/{name}", 16000, (row["audio"]["array"] * 32767).astype(np.int16))

NOISE_DIRS = [f"{SC_DIR}/_background_noise_"]

def to_16k_wavs(src_files, out_dir, desc):
    os.makedirs(out_dir, exist_ok=True)
    ds = datasets.Dataset.from_dict({"audio": [str(f) for f in src_files]}).cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(ds, desc=desc):
        name = Path(row["audio"]["path"]).stem + ".wav"
        scipy.io.wavfile.write(f"{out_dir}/{name}", 16000, (row["audio"]["array"] * 32767).astype(np.int16))

if DOWNLOAD_EXTRA_NOISE:
    if not os.path.isdir(f"{WORK}/audioset_16k"):
        os.makedirs(f"{WORK}/audioset", exist_ok=True)
        !wget -q --show-progress -O {WORK}/audioset/bal_train09.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
        !cd {WORK}/audioset && tar -xf bal_train09.tar
        to_16k_wavs(Path(f"{WORK}/audioset/audio").glob("**/*.flac"), f"{WORK}/audioset_16k", "AudioSet -> 16 kHz")
    if not os.path.isdir(f"{WORK}/fma_16k"):
        os.makedirs(f"{WORK}/fma", exist_ok=True)
        !wget -q --show-progress -O {WORK}/fma/fma_xs.zip https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip
        !cd {WORK}/fma && unzip -q -o fma_xs.zip
        to_16k_wavs(Path(f"{WORK}/fma/fma_small").glob("**/*.mp3"), f"{WORK}/fma_16k", "FMA music -> 16 kHz")
    NOISE_DIRS += [f"{WORK}/audioset_16k", f"{WORK}/fma_16k"]

print("RIR files:", len(os.listdir(RIR_DIR)))
for d in NOISE_DIRS:
    print("noise:", d, len(glob.glob(f"{d}/*.wav")), "files")

## Step 7: Set up augmentation and listen to one example
Each clip gets placed at the end of a 3.2 s window, with random room echo, background noise (−5 to +10 dB SNR), volume changes, EQ, a slight pitch shift and distortion. **Listen:** you should still (just about) hear the keyword.

In [ ]:
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from microwakeword.audio.audio_utils import save_clip
from IPython.display import Audio, display

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        "SevenBandParametricEQ": 0.1, "TanhDistortion": 0.1, "PitchShift": 0.1, "BandStopFilter": 0.1,
        "AddColorNoise": 0.1, "AddBackgroundNoise": 0.75, "Gain": 1.0, "RIR": 0.5,
    },
    impulse_paths=[RIR_DIR],
    background_paths=NOISE_DIRS,
    background_min_snr_db=-5,
    background_max_snr_db=10,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)
preview = Clips(input_directory=f"{POS_DIR}/training", file_pattern="*.wav")
save_clip(augmenter.augment_clip(preview.get_random_clip()), f"{WORK}/augmented_example.wav")
display(Audio(f"{WORK}/augmented_example.wav"))

## Step 8: Turn audio into features (≈10–30 min)
The ESP32 never runs the model on raw audio. Every 10 ms it computes **40 numbers** (a compressed spectrogram with noise suppression and automatic gain) from the last 30 ms of sound, using TensorFlow's `micro_frontend`. We compute exactly the same features here with `pymicro-features`, the identical C code, so training and the device match.

Positive clips are augmented twice for training, and each one is also saved at 10 slightly shifted positions to mimic streaming.

In [ ]:
import hashlib
from mmap_ninja.ragged import RaggedMmap

def make_features(clips_dir, out_root, name, split, repeat, slide_frames):
    out = f"{out_root}/{split}/{name}_mmap"
    files = sorted(glob.glob(f"{clips_dir}/*.wav"))
    sig = hashlib.md5(f"{[os.path.basename(f) for f in files]}|{repeat}|{slide_frames}".encode()).hexdigest()
    if os.path.isdir(out) and os.path.exists(out + ".sig") and open(out + ".sig").read() == sig:
        print("up to date, skipping:", out); return
    shutil.rmtree(out, ignore_errors=True)    # clips changed since last time -> rebuild
    if not files:
        print("no clips, skipping:", clips_dir); return
    os.makedirs(f"{out_root}/{split}", exist_ok=True)
    tmp = out + "_unfinished"                 # renamed only when complete, so a crash never leaves half a set
    shutil.rmtree(tmp, ignore_errors=True)
    clips = Clips(input_directory=clips_dir, file_pattern="*.wav")
    gen = SpectrogramGeneration(clips=clips, augmenter=augmenter, slide_frames=slide_frames, step_ms=10)
    print(f"{name}/{split}: {len(files)} clips x{repeat}" + (f" x{slide_frames} shifts" if slide_frames else ""))
    RaggedMmap.from_generator(out_dir=tmp, sample_generator=gen.spectrogram_generator(repeat=repeat),
                              batch_size=100, verbose=True)
    os.rename(tmp, out)
    open(out + ".sig", "w").write(sig)

# positives: same recipe as the official microWakeWord notebook
make_features(f"{POS_DIR}/training",   FEAT_POS, "wakeword", "training",   repeat=2, slide_frames=10)
make_features(f"{POS_DIR}/validation", FEAT_POS, "wakeword", "validation", repeat=1, slide_frames=10)
make_features(f"{POS_DIR}/testing",    FEAT_POS, "wakeword", "testing",    repeat=1, slide_frames=1)
# extra negatives (other Speech Commands words)
for s in ("training", "validation", "testing"):
    make_features(f"{NEG_DIR}/{s}", FEAT_NEG, "sc_negatives", s, repeat=1, slide_frames=None)
# hard negatives (sound-alikes + the ESP32's false triggers): 2 different augmentations of each for training
make_features(f"{HARD_DIR}/training",   FEAT_HARD, "hard_negatives", "training",   repeat=2, slide_frames=None)
make_features(f"{HARD_DIR}/validation", FEAT_HARD, "hard_negatives", "validation", repeat=1, slide_frames=None)
make_features(f"{HARD_DIR}/testing",    FEAT_HARD, "hard_negatives", "testing",    repeat=1, slide_frames=None)

## Step 9: Download the big ready-made negative feature sets (≈5.7 GB, 5–15 min)
These are already-computed features from hours of **speech** (LibriSpeech and others), **dinner-party conversations** (CHiME-6) and **non-speech** audio (music and noise). `dinner_party_eval` is used only to measure **false accepts per hour**.

In [ ]:
os.makedirs(HF_NEG, exist_ok=True)
for fname in ["dinner_party.zip", "dinner_party_eval.zip", "no_speech.zip", "speech.zip"]:
    if os.path.isdir(f"{HF_NEG}/{fname[:-4]}"):
        print("exists:", fname); continue
    !wget -q --show-progress -O {HF_NEG}/{fname} https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/{fname}
    !unzip -q -o {HF_NEG}/{fname} -d {HF_NEG} && rm {HF_NEG}/{fname}
!du -sh {HF_NEG}/*

## Step 10: Write the training configuration
* `sampling_weight`: how often each set is picked for a training batch.
* `penalty_weight` / `negative_class_weight`: how much a mistake on that set costs. Raising the negative weight gives **fewer false triggers** but more missed words.
* `clip_duration_ms: 1500`: the longest keyword the model can "see" at once. Keep it for keywords under about 1.2 s.

In [ ]:
import yaml
config = {
    "window_step_ms": 10,
    "train_dir": TRAIN_DIR,
    "features": [
        {"features_dir": FEAT_POS, "sampling_weight": 2.0, "penalty_weight": 1.0, "truth": True,
         "truncation_strategy": "truncate_start", "type": "mmap"},
        {"features_dir": FEAT_NEG, "sampling_weight": 5.0, "penalty_weight": 1.0, "truth": False,
         "truncation_strategy": "truncate_start", "type": "mmap"},
        {"features_dir": f"{HF_NEG}/speech", "sampling_weight": 10.0, "penalty_weight": 1.0, "truth": False,
         "truncation_strategy": "random", "type": "mmap"},
        {"features_dir": f"{HF_NEG}/dinner_party", "sampling_weight": 10.0, "penalty_weight": 1.0, "truth": False,
         "truncation_strategy": "random", "type": "mmap"},
        {"features_dir": f"{HF_NEG}/no_speech", "sampling_weight": 5.0, "penalty_weight": 1.0, "truth": False,
         "truncation_strategy": "random", "type": "mmap"},
        {"features_dir": f"{HF_NEG}/dinner_party_eval", "sampling_weight": 0.0, "penalty_weight": 1.0, "truth": False,
         "truncation_strategy": "split", "type": "mmap"},   # only for measuring false accepts per hour
    ],
    "training_steps": [TRAINING_STEPS],
    "positive_class_weight": [1],
    "negative_class_weight": [20],
    "learning_rates": [0.001],
    "batch_size": 128,
    "time_mask_max_size": [0], "time_mask_count": [0],   # SpecAugment (off to start)
    "freq_mask_max_size": [0], "freq_mask_count": [0],
    "eval_step_interval": min(500, TRAINING_STEPS),
    "clip_duration_ms": 1500,
    "target_minimization": 0.9,
    "minimization_metric": None,
    "maximization_metric": "average_viable_recall",
}
if os.path.isdir(f"{FEAT_HARD}/training"):     # the sound-alikes, with their own weight (Step 2: HARD_NEG_WEIGHT)
    config["features"].insert(2, {"features_dir": FEAT_HARD, "sampling_weight": HARD_NEG_WEIGHT, "penalty_weight": 1.0,
                                  "truth": False, "truncation_strategy": "truncate_start", "type": "mmap"})
CONFIG_PATH = f"{OUT}/training_parameters.yaml"
with open(CONFIG_PATH, "w") as f:
    yaml.dump(config, f)
print("Saved", CONFIG_PATH)

## Step 10b: RAM fix (run right before training)
Every 500 steps the training checks false triggers on hours of dinner-party audio. The original code builds all
of it at once as one large array (plus two copies), which makes Colab run out of RAM. This cell patches that check
(on this Colab machine only) to feed the audio in chunks. The reported numbers, and therefore which checkpoint is
kept, are exactly the same. Tested against the pinned microWakeWord version: identical metrics, a fraction of the RAM.

In [ ]:
# RAM fix: run this right before Step 11 (after Step 10). Safe to run again; after Step 1 on a fresh machine, run it again.
#
# Why Colab runs out of RAM: every 500 training steps, microWakeWord checks false triggers on the dinner-party
# recordings by cutting them into 2-second windows, one every 300 ms, builds ALL of them as one float32 array and
# copies it twice more. That spike (30 times in a 15,000-step run) is what fills the RAM.
# This patch gives those windows to the model in chunks instead. The check already adds its counts up across
# calls (it switches the metric reset off there), so every number it reports, and therefore which checkpoint
# is kept as "best", is exactly the same. Only the peak RAM drops.
import os, gc, glob, ctypes, random, importlib.util, py_compile
import psutil, yaml
from mmap_ninja.ragged import RaggedMmap

CHUNK = 4096   # windows per chunk: 4096 x 204 x 40 x 4 bytes = 134 MB

# 1) patch the check inside the installed microWakeWord (on this Colab machine only)
TRAIN_PY = os.path.join(list(importlib.util.find_spec("microwakeword").submodule_search_locations)[0], "train.py")
src = open(TRAIN_PY).read()
if "RAM fix (notebook)" not in src:
    start = src.index('    if data_processor.get_mode_size("validation_ambient") > 0:\n')
    start = src.index("\n", start) + 1
    end = src.index("        duration_of_ambient_set = (")
    src = src[:start] + f"""        # RAM fix (notebook): evaluate the ambient windows in chunks of {CHUNK} instead of one huge array.
        # reset_metrics is swapped out, so the metrics accumulate over the chunks: same result as one call.
        def ambient_chunks():
            data, labels = [], []
            for provider in data_processor.feature_providers:
                for spectrogram in provider.get_feature_generator(
                    test_set + "_ambient", config["spectrogram_length"], "split"
                ):
                    data.append(spectrogram)
                    labels.append(provider.label)
                    if len(data) == {CHUNK}:
                        yield np.array(data), np.array(labels, np.float32).reshape(-1, 1)
                        data, labels = [], []
            if data:
                yield np.array(data), np.array(labels, np.float32).reshape(-1, 1)

        with swap_attribute(model, "reset_metrics", lambda: None):
            for chunk, truth in ambient_chunks():
                ambient_predictions = model.evaluate(
                    chunk, truth, batch_size=1024, return_dict=True, verbose=0
                )
                del chunk, truth

""" + src[end:]
    open(TRAIN_PY, "w").write(src)
py_compile.compile(TRAIN_PY, doraise=True)
print("patched:", TRAIN_PY)

# 2) how big that array really was (the training log's input_layer is 204 frames; a window every 30 frames)
L, HOP = 204, 30
windows = 0
for fs in yaml.safe_load(open(CONFIG_PATH))["features"]:
    for p in glob.glob(f"{fs['features_dir']}/validation_ambient/**/*_mmap", recursive=True):
        rm = RaggedMmap(p)
        windows += sum(len(range(0, rm[i].shape[0] - L, HOP)) for i in range(len(rm)))
full = windows * L * 40 * 4 / 1e9
print(f"background check: {windows:,} windows = {full:.1f} GB as one array; before this fix it was built and copied "
      f"(~{3 * full:.1f} GB at once) every 500 steps. Now {CHUNK * L * 40 * 4 / 1e6:.0f} MB per chunk.")

# 3) after a runtime restart: rebuild the same split of the ESP32's false triggers that Step 5e made (for Step 12b)
if "DEVICE_NEG_SPLIT" not in globals():
    dev = sorted(glob.glob(f"{WORK}/device_negatives/**/*.wav", recursive=True))
    random.Random(0).shuffle(dev)
    a = int(len(dev) * 0.6); b = a + int(len(dev) * 0.2)
    DEVICE_NEG_SPLIT = {"training": dev[:a], "validation": dev[a:b], "testing": dev[b:]}

# 4) give this notebook's own unused memory back, so the training process can use it
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)
vm, me = psutil.virtual_memory(), psutil.Process().memory_info().rss
print(f"RAM: {vm.available / 1e9:.1f} of {vm.total / 1e9:.1f} GB available; this notebook itself holds {me / 1e9:.1f} GB")
if me > 1.5e9:
    print("  For more headroom: Runtime > Restart session, then run only Steps 2, 2b, 3, 10, this cell and Step 11 "
          "(everything else is already on disk).")

## Step 11: Train, then convert to a streaming int8 TFLite model (roughly 1–2 h for 10,000 steps on a T4)
* The model is **MixedNet**: small depthwise convolutions of several sizes, designed for streaming.
* It trains on whole 1.5 s windows, then gets converted to a **streaming** model that keeps its own internal state and only takes the newest 3 feature frames (30 ms) per call. That's what makes it cheap on a microcontroller.
* Colab may print nothing for several minutes at a time. That's normal. Validation results appear every `eval_step_interval` steps.
* If Colab disconnects, the machine is wiped (installs, datasets, features) but the checkpoints are safe in Drive: run every step again from Step 1 (the downloads repeat, Step 2b re-uploads), then this cell. Training continues where it stopped. It resumes from the last checkpoint in Drive.

In [ ]:
TRAIN_CMD = (
    f"cd /content && {sys.executable} -m microwakeword.model_train_eval "
    f"--training_config='{CONFIG_PATH}' --train 1 --restore_checkpoint 1 "
    "--test_tf_nonstreaming 0 --test_tflite_nonstreaming 0 --test_tflite_nonstreaming_quantized 0 "
    "--test_tflite_streaming 0 --test_tflite_streaming_quantized 1 --use_weights best_weights "
    "mixednet --pointwise_filters '64,64,64,64' --repeat_in_block '1,1,1,1' "
    "--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' --residual_connection '0,0,0,0' "
    "--first_conv_filters 32 --first_conv_kernel_size 5 --stride 3"
)
!{TRAIN_CMD}

## Step 12: Read the results and pick the detection threshold
The test runs the **final int8 streaming model** exactly as the ESP32 will:
* **frr** (false rejection rate): the share of real keyword utterances it missed. Lower is better.
* **faph** (false accepts per hour): false triggers per hour of background conversation. Lower is better.
* **Cutoff**: the probability threshold. A higher cutoff means fewer false triggers but more misses.

As a starting target, aim for **faph ≤ 0.5** with frr as low as possible. For SIH ("near-zero false activations"), pick the lowest cutoff where faph is 0 or close to it.

In [ ]:
import matplotlib.pyplot as plt
ROC = f"{TRAIN_DIR}/tflite_stream_state_internal_quant/tflite_streaming_roc.txt"
rows = []
for line in open(ROC):
    m = re.match(r"Cutoff ([\d.]+): frr=([\d.]+); faph=([\d.]+)", line)
    if m: rows.append(tuple(float(x) for x in m.groups()))
print(open(ROC).readline().strip(), "(area under the miss-vs-false-trigger curve; smaller is better)")
print(f"{'cutoff':>7} {'missed (frr)':>13} {'false/hour':>11}")
for c, frr, faph in rows:
    print(f"{c:7.2f} {frr:13.1%} {faph:11.2f}")

TARGET_FAPH = 0.5
ok = [r for r in rows if r[2] <= TARGET_FAPH]
best = min(ok, key=lambda r: (r[1], r[0])) if ok else max(rows, key=lambda r: r[0])
PROBABILITY_CUTOFF = round(max(best[0], 0.5), 2)
print(f"\n→ Suggested probability_cutoff = {PROBABILITY_CUTOFF}  (at cutoff {best[0]:.2f}: misses {best[1]:.1%}, {best[2]:.2f} false triggers/hour)")
if best[0] < 0.5:
    print("  (raised to 0.5: a cutoff below 0.5 usually means the model needs more training/data)")

plt.figure(figsize=(5, 3.5))
plt.plot([r[2] for r in rows], [r[1] for r in rows], marker=".")
plt.xlabel("false accepts per hour"); plt.ylabel("false rejection rate"); plt.title(f"'{WAKE_WORD}': lower-left is better")
plt.grid(alpha=0.3); plt.show()

## Step 12b: The real test: old vs new model on the ESP32's own recordings
Runs both models on the device's recordings **exactly the way the firmware does** (same features, int8 input,
3-frame streaming, 5-value sliding average). Each recording triggered the ESP32, so for the old model you should
see nearly all of them fire; that confirms the test matches the device.
* **Real "Marvin"** (the ESP32's microphone, never used for training): should still fire. This is the miss rate on real use.
* **False triggers the model never trained on**: the honest number. Should drop close to 0.
* **False triggers it trained on**: should be 0 (if not, the model hasn't learned them; raise `HARD_NEG_WEIGHT`).

The table uses `PROBABILITY_CUTOFF` from Step 12. The sweep below it shows other thresholds. To use a different
one, run `PROBABILITY_CUTOFF = 0.8` (for example) in a new cell before Step 13.

In [ ]:
import numpy as np, soundfile as sf
from pymicro_features import MicroFrontend
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    from tensorflow.lite.python.interpreter import Interpreter

NEW_MODEL = f"{TRAIN_DIR}/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"

def peak_score(model_path, clip, window=5):
    """Highest sliding-window probability for one recording, fed exactly like the ESP32 firmware does:
    microfrontend features every 10 ms -> int8 -> 3 frames per call -> average of the last 5 outputs."""
    it = Interpreter(model_path=model_path)
    it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    stride = int(inp["shape"][1])
    (s_in, z_in), (s_out, z_out) = inp["quantization"], out["quantization"]
    noise = np.random.default_rng(1).integers(-30, 31, 28800).astype(np.int16)   # quiet noise, about -66 dBFS
    audio = np.concatenate([noise[:19200], clip, noise[19200:]])                  # 1.2 s before, 0.6 s after
    fe, frames, probs, best, pos = MicroFrontend(), [], [0.0] * window, 0.0, 0
    while pos + 160 <= len(audio):
        r = fe.process_samples(audio[pos:pos + 160].tobytes())
        pos += max(r.samples_read, 1)
        if not r.features:
            continue
        f = np.asarray(r.features, np.float32)                   # already scaled like the firmware (x 0.0390625)
        frames.append(np.clip(np.rint(f / s_in) + z_in, -128, 127).astype(np.int8))
        if len(frames) == stride:
            it.set_tensor(inp["index"], np.stack(frames)[None, ...])
            frames = []
            it.invoke()
            probs = probs[1:] + [(int(it.get_tensor(out["index"])[0][0]) - z_out) * s_out]
            best = max(best, sum(probs) / window)
    return best

def load_clip(path):
    x, _ = sf.read(path, dtype="int16", always_2d=True)
    return x[:, 0]

groups = {
    'real "Marvin" (ESP32 mic)':        sorted(glob.glob(f"{WORK}/my_recordings/**/*.wav", recursive=True)),
    "false triggers NOT trained on":    sorted(DEVICE_NEG_SPLIT["validation"] + DEVICE_NEG_SPLIT["testing"]),
    "false triggers trained on":        sorted(DEVICE_NEG_SPLIT["training"]),
}
models = {"old": OLD_MODEL, "new": NEW_MODEL} if OLD_MODEL else {"new": NEW_MODEL}
clips = {g: [load_clip(f) for f in files] for g, files in groups.items()}
peaks = {m: {g: [peak_score(p, c) for c in cs] for g, cs in clips.items()} for m, p in models.items()}

cut = PROBABILITY_CUTOFF
print(f"Firing at threshold {cut:.2f} (from Step 12). Want: real Marvin HIGH, false triggers LOW.\n")
print(f"{'':34}" + "".join(f"{m + ' model':>18}" for m in models))
for g, files in groups.items():
    if not files:
        print(f"{g:34}" + "".join(f"{'(none uploaded)':>18}" for _ in models))
        continue
    cells = [f"{sum(p > cut for p in peaks[m][g])}/{len(files)} ({100 * sum(p > cut for p in peaks[m][g]) / len(files):.0f}%)" for m in models]
    print(f"{g:34}" + "".join(f"{c:>18}" for c in cells))

pos_g, held_g = 'real "Marvin" (ESP32 mic)', "false triggers NOT trained on"
if clips[pos_g] or clips[held_g]:
    print("\nNew model at other thresholds (real Marvin detected / unseen false triggers fired):")
    for c in np.arange(0.50, 0.96, 0.05):
        det = sum(p > c for p in peaks["new"][pos_g]); fa = sum(p > c for p in peaks["new"][held_g])
        print(f"  {c:.2f}: {det}/{len(clips[pos_g])} detected, {fa}/{len(clips[held_g])} false triggers")

print("\nPer recording (peak score, old -> new):")
for g, files in groups.items():
    for i, f in enumerate(files):
        old = f"{peaks['old'][g][i]:.2f} -> " if "old" in peaks else ""
        print(f"  {g[:22]:22} {os.path.basename(f)[:30]:30} {old}{peaks['new'][g][i]:.2f}")

## Step 13: Export for the ESP32
You get:
* `<name>.tflite`: the int8 streaming model.
* `<name>.json`: a model manifest in ESPHome's v2 format (threshold, window size, etc.).
* `<name>_model_data.h` / `.cc`: the model as a C array, ready to compile into an ESP-IDF project with `esp-tflite-micro`.
* `ops.txt`: the list of TFLite operators the firmware must register.

In [ ]:
EXPORT = f"{OUT}/export"
os.makedirs(EXPORT, exist_ok=True)
TFLITE = f"{TRAIN_DIR}/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"
shutil.copy(TFLITE, f"{EXPORT}/{MODEL_NAME}.tflite")
model_bytes = open(TFLITE, "rb").read()
print(f"Model size: {len(model_bytes)/1024:.1f} KB (flash)")

manifest = {
    "type": "micro", "wake_word": WAKE_WORD, "author": "SIH team", "website": "",
    "model": f"{MODEL_NAME}.tflite", "trained_languages": ["en"], "version": 2,
    "micro": {"probability_cutoff": PROBABILITY_CUTOFF, "sliding_window_size": 5, "feature_step_size": 10,
              "tensor_arena_size": 30000, "minimum_esphome_version": "2024.7.0"},
}
json.dump(manifest, open(f"{EXPORT}/{MODEL_NAME}.json", "w"), indent=2)

# C array for ESP-IDF (16-byte aligned, as TFLite Micro requires)
hex_lines = ",\n".join("  " + ", ".join(f"0x{b:02x}" for b in model_bytes[i:i+12]) for i in range(0, len(model_bytes), 12))
open(f"{EXPORT}/{MODEL_NAME}_model_data.h", "w").write(
    f"#pragma once\n#include <cstdint>\nextern const unsigned char g_{MODEL_NAME}_model[];\nextern const unsigned int g_{MODEL_NAME}_model_len;\n"
    f"#define WAKE_WORD_PROBABILITY_CUTOFF {PROBABILITY_CUTOFF}f\n#define WAKE_WORD_SLIDING_WINDOW 5\n")
open(f"{EXPORT}/{MODEL_NAME}_model_data.cc", "w").write(
    f'#include "{MODEL_NAME}_model_data.h"\nalignas(16) const unsigned char g_{MODEL_NAME}_model[] = {{\n{hex_lines}\n}};\n'
    f"const unsigned int g_{MODEL_NAME}_model_len = {len(model_bytes)};\n")

# operators used (the firmware's MicroMutableOpResolver must add exactly these)
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    from tensorflow.lite.python.interpreter import Interpreter
it = Interpreter(model_path=TFLITE)
ops = sorted({d["op_name"] for d in it._get_ops_details()})
inp, outp = it.get_input_details()[0], it.get_output_details()[0]
info = (f"ops ({len(ops)}): {', '.join(ops)}\n"
        f"input : {inp['shape'].tolist()} {inp['dtype'].__name__} scale={inp['quantization'][0]} zero_point={inp['quantization'][1]}\n"
        f"output: {outp['shape'].tolist()} {outp['dtype'].__name__} scale={outp['quantization'][0]} zero_point={outp['quantization'][1]}\n")
open(f"{EXPORT}/ops.txt", "w").write(info); print(info)

shutil.make_archive(f"/content/{MODEL_NAME}_export", "zip", EXPORT)
print("Exported to", EXPORT)
try:
    from google.colab import files
    files.download(f"/content/{MODEL_NAME}_export.zip")
except Exception:
    pass

## Put the new model on the ESP32
1. Unzip `marvin_export.zip` (downloaded above; also in Drive under `microwakeword/marvin_v2/export`).
2. Copy **`marvin.tflite`** and **`marvin.json`** into `kws_s3_firmware/kws_s3/model/`, replacing the old ones
   (keep a copy of the old ones if you want to switch back).
3. Rebuild and flash: `ninja -C build -j 3` then `idf.py -p COM6 flash` (or ask Claude to do it). The boot log shows
   the new threshold: `model ready: wake word "marvin", threshold 0.xx (json)`.
4. Test with the same words as before ("Marvin, ...", "Martin", "marble", "Markovnikov", "carving").

## Keep improving (the loop)
Every utterance the ESP32 streams is saved and labelled by the cloud server, so the data grows by itself:
1. Use the device normally for a while (and say some sound-alikes on purpose).
2. On the PC: `python kws_s3/tools/export_training_clips.py cloudServer/server/recordings kws_s3/recordings --stt http://127.0.0.1:8000/transcribe --relabel`
3. Listen to a few clips in `kws_s3/training_clips/` (Whisper can mishear a distant "Marvin"), delete wrong ones, re-zip.
4. Run this notebook again with a new `RUN_NAME` (e.g. `marvin_v3`) and compare in Step 12b.

Once you have **100+ real "Marvin" recordings** from the ESP32, the notebook automatically uses 60% of them for training too
(below 100 they are used only for testing), which is the single biggest improvement for the real microphone.